# secrets_manager

**Source:** `00_common/secrets_manager.py`  
**Purpose:** Databricks notebook auto-generated from framework Python module.


## Section 1: Additional module code and configuration

This cell handles: *Additional module code and configuration*


In [ ]:
"""Secrets management integration with Databricks Secrets API."""

from __future__ import annotations

import logging
import os
from abc import ABC, abstractmethod
from typing import Optional

logger = logging.getLogger(__name__)


## Section 2: Define `SecretsBackend` class

This cell handles: *Define `SecretsBackend` class*


In [ ]:
class SecretsBackend(ABC):
    """Abstract base class for secrets backends."""
    
    @abstractmethod


## Section 3: Define `get_secret()` function with logic for processing

This cell handles: *Define `get_secret()` function with logic for processing*


In [ ]:
    def get_secret(self, key: str, default: Optional[str] = None) -> Optional[str]:
        """Retrieve a secret."""
        pass


## Section 4: Define `EnvironmentSecretsBackend` class

This cell handles: *Define `EnvironmentSecretsBackend` class*


In [ ]:
class EnvironmentSecretsBackend(SecretsBackend):
    """Secrets from environment variables."""


## Section 5: Define `get_secret()` function with logic for processing

This cell handles: *Define `get_secret()` function with logic for processing*


In [ ]:
    def get_secret(self, key: str, default: Optional[str] = None) -> Optional[str]:
        """Get secret from environment variable."""
        return os.getenv(key, default)


## Section 6: Define `DatabricksSecretsBackend` class

This cell handles: *Define `DatabricksSecretsBackend` class*


In [ ]:
class DatabricksSecretsBackend(SecretsBackend):
    """
    Secrets from Databricks Secrets API.
    
    Requires spark context and Databricks Secrets Scope configured.
    """


## Section 7: Define `__init__()` function with logic for processing

This cell handles: *Define `__init__()` function with logic for processing*


In [ ]:
    def __init__(self, scope: str = "default", spark=None):
        """
        Initialize Databricks Secrets Backend.
        
        Args:
            scope: Secret scope name (default: "default")
            spark: Spark session for API access
        """
        self.scope = scope
        self.spark = spark


## Section 8: Define `get_secret()` function with logic for processing

This cell handles: *Define `get_secret()` function with logic for processing*


In [ ]:
    def get_secret(self, key: str, default: Optional[str] = None) -> Optional[str]:
        """Get secret from Databricks Secrets API."""
        if not self.spark:
            logger.warning("Spark session not available, falling back to environment variable")
            return os.getenv(key, default)
        
        try:
            # Use Databricks Secrets API via dbutils
            dbutils = self.spark.sparkContext._jvm.com.databricks.python.PythonUtils.getDBUtils(self.spark.sparkContext)
            secret = dbutils.secrets.get(scope=self.scope, key=key)
            logger.debug(f"Retrieved secret from Databricks Secrets: {self.scope}.{key}")
            return secret
        except Exception as e:
            logger.warning(f"Failed to retrieve secret from Databricks: {e}, using default")
            return default


## Section 9: Define `SecretsManager:` class

This cell handles: *Define `SecretsManager:` class*


In [ ]:
class SecretsManager:
    """
    Central secrets manager with pluggable backends.
    
    Supports environment variables, Databricks Secrets, and custom backends.
    """


## Section 10: Define `__init__()` function with logic for processing

This cell handles: *Define `__init__()` function with logic for processing*


In [ ]:
    def __init__(self, backend: Optional[SecretsBackend] = None):
        """
        Initialize SecretsManager.
        
        Args:
            backend: Secrets backend (default: environment variables)
        """
        self.backend = backend or EnvironmentSecretsBackend()


## Section 11: Define `get_secret()` function with logic for processing

This cell handles: *Define `get_secret()` function with logic for processing*


In [ ]:
    def get_secret(self, key: str, default: Optional[str] = None) -> Optional[str]:
        """
        Retrieve a secret.
        
        Args:
            key: Secret key/name
            default: Default value if secret not found
            
        Returns:
            Secret value or default
        """
        secret = self.backend.get_secret(key, default)
        if not secret and not default:
            logger.warning(f"Secret not found: {key}")
        return secret


## Section 12: Define `get_required_secret()` function with logic for processing

This cell handles: *Define `get_required_secret()` function with logic for processing*


In [ ]:
    def get_required_secret(self, key: str) -> str:
        """
        Retrieve a required secret.
        
        Raises:
            ValueError: If secret not found
        """
        secret = self.get_secret(key)
        if not secret:
            raise ValueError(f"Required secret not found: {key}")
        return secret


## Section 13: Define `configure_secrets_from_global_config()` function with logic for processing

This cell handles: *Define `configure_secrets_from_global_config()` function with logic for processing*


In [ ]:
def configure_secrets_from_global_config(
    global_config: dict,
    spark=None,
) -> SecretsManager:
    """
    Configure SecretsManager from global_config.
    
    Args:
        global_config: Global configuration dict
        spark: Spark session (required for Databricks backend)
        
    Returns:
        Configured SecretsManager
    """
    secrets_config = global_config.get("secrets", {})
    backend_type = secrets_config.get("backend", "env_vars").lower()
    
    if backend_type == "databricks_secrets":
        scope = secrets_config.get("databricks_scope", "default")
        backend = DatabricksSecretsBackend(scope=scope, spark=spark)
        logger.info(f"Using Databricks Secrets backend (scope={scope})")
    else:
        backend = EnvironmentSecretsBackend()
        logger.info("Using environment variables backend")
    
    return SecretsManager(backend=backend)
